# Walton SOP - Strategy Research Workflow

This notebook demonstrates the full research pipeline for evaluating a trading idea.

## The Process (in order of importance)

1. **State the thesis** - What structural reason should this edge persist?
2. **Fetch data** - Get OHLCV data with proper train/val/test splits
3. **Backtest on training set** - Initial signal check
4. **Robustness checks** - Parameter sensitivity, Monte Carlo, regime analysis
5. **Walk-forward analysis** - The real test of out-of-sample validity
6. **Verdict** - Does this survive scrutiny or is it noise?

## Critical Mindset

The default assumption is that **any strategy is noise until proven otherwise**.
We're looking for reasons to REJECT a strategy, not confirm it.
If it survives all the stress tests, it *might* have a real edge.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from walton.data import fetch_ohlcv, make_splits
from walton.engine import run_backtest
from walton.walkforward import run_walk_forward
from walton.robustness import (
    test_parameter_sensitivity,
    monte_carlo_test,
    analyze_regimes,
)
from walton.viz import (
    plot_equity_curve,
    plot_returns_distribution,
    plot_trade_analysis,
    plot_parameter_sensitivity,
    plot_monte_carlo,
    plot_regime_breakdown,
    plot_walk_forward,
    plot_comparison,
)

# Import strategies
from walton.strategies.mean_reversion import BollingerMeanReversion
from walton.strategies.trend_following import DualMACrossover, DonchianBreakout

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print('Framework loaded.')

## Step 1: State the Thesis

Before looking at ANY data, write down:
1. What is the structural reason this should work?
2. In what market regime should it profit?
3. When should it FAIL? (If you can't answer this, the thesis is too vague)

**Example thesis for Mean Reversion:**
- *Why:* Short-term overreaction to news creates temporary mispricings
- *When it works:* Range-bound, liquid markets with mean-reverting behavior
- *When it fails:* Strong trends, crashes, regime shifts
- *Edge source:* Providing liquidity when others panic

## Step 2: Fetch Data & Create Splits

In [ ]:
# Fetch data - use a long history for statistical significance
symbol = 'SPY'
df = fetch_ohlcv(symbol, start='2005-01-01', end='2024-12-31', cache_dir='../data/cache')

print(f'Data: {symbol}')
print(f'Period: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Bars: {len(df)}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Create train/val/test splits
# 60% train, 20% validation, 20% final test
# IMPORTANT: Never look at test set until ALL decisions are final
splits = make_splits(df, train_frac=0.6, val_frac=0.2, test_frac=0.2)

for name, split in splits.items():
    print(f'{name}: {split}')

## Step 3: Initial Backtest on Training Data

Run the strategy on the TRAINING set only. This is where we develop intuition.
Do NOT touch the test set yet.

In [ ]:
# Choose a strategy to evaluate
strategy = BollingerMeanReversion()
params = strategy.default_params()

print(f'Strategy: {strategy.name}')
print(f'Parameters: {params}')
print(f'\nThesis check:')
print(f'  - Structural reason: Overreaction/liquidity provision')
print(f'  - Expected regime: Low-vol, choppy markets')
print(f'  - Expected failure: Strong trends, crashes')

In [ ]:
# Backtest on training data
train_result = run_backtest(
    strategy, 
    splits['train'].df, 
    params,
    commission_bps=5.0,   # Realistic commission
    slippage_bps=5.0,     # Realistic slippage
    split_name='train',
)

# Print summary
for k, v in train_result.summary().items():
    print(f'  {k}: {v}')

# Check minimum bar
passes, issues = train_result.passes_minimum_bar(min_trades=50)
if not passes:
    print('\n⚠ WARNINGS:')
    for issue in issues:
        print(f'  - {issue}')
else:
    print('\nPasses minimum statistical bar.')

In [ ]:
# Visualize training results
fig = plot_equity_curve(train_result, benchmark=splits['train'].df['Close'])
plt.show()

fig = plot_returns_distribution(train_result)
plt.show()

fig = plot_trade_analysis(train_result)
plt.show()

## Step 4: Robustness Checks

These are the critical tests that separate real edges from noise.

### 4a. Parameter Sensitivity
If changing a parameter by 10-20% kills the strategy, it's overfit to that exact value.

In [ ]:
# Test parameter sensitivity on training data
perturbation_results = test_parameter_sensitivity(
    strategy, splits['train'].df, params,
    perturbation_levels=[-0.3, -0.2, -0.1, 0.1, 0.2, 0.3],
)

for pr in perturbation_results:
    status = 'ROBUST' if pr.is_robust else 'FRAGILE'
    print(f'{pr.param_name}: sensitivity={pr.sensitivity:.3f} [{status}]')

fig = plot_parameter_sensitivity(perturbation_results)
plt.show()

### 4b. Monte Carlo Permutation Test

Shuffle the order of trades. If random orderings produce similar results,
the sequence of trades doesn't matter - meaning the strategy might just be
capturing random variance rather than a real pattern.

In [ ]:
mc_result = monte_carlo_test(train_result, n_simulations=1000)

print(f'Actual Sharpe: {mc_result.actual_sharpe:.2f}')
print(f'p-value: {mc_result.p_value:.3f}')
print(f'Percentile: {mc_result.sharpe_percentile:.0f}th')

if mc_result.p_value < 0.05:
    print('\nResult: Statistically significant (p < 0.05)')
else:
    print('\nResult: NOT statistically significant - likely noise')

fig = plot_monte_carlo(mc_result)
plt.show()

### 4c. Regime Analysis

A robust strategy should have a clear thesis about which regime it profits in.
If it makes money uniformly across all regimes, be suspicious - it might be
an artifact. If it makes money in the regime it SHOULD (per thesis) and loses
in others, that's actually a good sign of a real, explainable edge.

In [ ]:
regime_results = analyze_regimes(train_result, splits['train'].df)

for r in regime_results:
    print(f'{r.regime_name}: Sharpe={r.sharpe:.2f}, Return={r.total_return:.2%}, '
          f'Trades={r.n_trades}, WinRate={r.win_rate:.1%}')

fig = plot_regime_breakdown(regime_results)
plt.show()

print('\nThesis check:')
print('  Mean reversion should profit in low-vol choppy regimes.')
print('  Does the regime breakdown match this expectation?')

## Step 5: Walk-Forward Analysis

This is the gold standard test. It simulates real-time optimization and trading:
1. Optimize on a training window
2. Trade the next out-of-sample window
3. Roll forward and repeat

Key metric: Walk-Forward Efficiency (WFE) = OOS performance / IS performance
- WFE > 0.5 is decent, > 0.7 is good
- WFE < 0.3 means the strategy is overfit

In [ ]:
# Define parameter grid for walk-forward optimization
param_grid = {
    'bb_period': [10, 15, 20, 25, 30],
    'bb_std': [1.5, 2.0, 2.5],
    'rsi_period': [10, 14, 20],
    'rsi_oversold': [25, 30, 35],
}

# Run walk-forward on the combined train+val data
# (Reserve test set for final confirmation only)
train_val_df = pd.concat([splits['train'].df, splits['val'].df])

wf_result = run_walk_forward(
    strategy,
    train_val_df,
    param_grid=param_grid,
    n_windows=5,
    train_ratio=0.7,
    optimize_metric='sharpe',
    commission_bps=5.0,
    slippage_bps=5.0,
)

# Print summary
print('Walk-Forward Results:')
for k, v in wf_result.summary().items():
    print(f'  {k}: {v}')

fig = plot_walk_forward(wf_result)
plt.show()

## Step 6: Final Verdict

Only after ALL the above checks, look at the test set.
This should be a one-time confirmation, not an iterative process.

In [ ]:
# Final test set evaluation (ONLY DO THIS ONCE)
test_result = run_backtest(
    strategy, splits['test'].df, params,
    commission_bps=5.0,
    slippage_bps=5.0,
    split_name='test',
)

print('=== FINAL VERDICT ===')
print()
print('Training set:')
for k, v in train_result.summary().items():
    print(f'  {k}: {v}')

print()
print('Test set (out-of-sample):')
for k, v in test_result.summary().items():
    print(f'  {k}: {v}')

print()
print('Walk-Forward Efficiency:', wf_result.summary()['avg_wfe'])

# Comparison plot
fig = plot_comparison([train_result, test_result])
plt.show()

In [ ]:
# Scorecard: Is this strategy worth pursuing?
print('=== STRATEGY SCORECARD ===')
print()

checks = [
    ('Sufficient trades (>100)', train_result.n_trades >= 100),
    ('Positive Sharpe in-sample', train_result.sharpe_ratio > 0),
    ('Positive Sharpe out-of-sample', test_result.sharpe_ratio > 0),
    ('WFE > 0.3', wf_result.avg_wfe > 0.3),
    ('Monte Carlo p < 0.05', mc_result.p_value < 0.05),
    ('Parameters are robust', all(pr.is_robust for pr in perturbation_results)),
    ('Max DD < 20%', abs(test_result.max_drawdown) < 0.20),
    ('Regime thesis confirmed', True),  # Manual check
]

for check_name, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status}] {check_name}')

n_passed = sum(1 for _, p in checks if p)
print(f'\nScore: {n_passed}/{len(checks)}')

if n_passed >= 7:
    print('Verdict: PROMISING - Worth further investigation with live paper trading')
elif n_passed >= 5:
    print('Verdict: MIXED - Some signal but significant concerns remain')
else:
    print('Verdict: REJECT - Insufficient evidence of a real edge')

## Adding Your Own Strategy

To test a new idea:

```python
from walton.strategy import Strategy, StrategyParams

class MyStrategy(Strategy):
    @property
    def name(self) -> str:
        return "My Strategy Name"

    def default_params(self) -> StrategyParams:
        params = StrategyParams()
        params.set("my_param", 20, lo=5, hi=100)
        return params

    def generate_signals(self, df, params):
        # Your signal logic here
        # Return Series of -1 to +1
        pass
```

Then run through the same pipeline above. The framework handles everything else.